# **Lab 4.1. Шаврин Алексей, группа 1306**

# Импорты

In [1]:
import numpy as np
import pandas as pd
import os
import tarfile
from six.moves import urllib
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit, GridSearchCV, RandomizedSearchCV
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from scipy.stats import expon, reciprocal
import matplotlib.pyplot as plt

np.random.seed(42)

# Загрузка исходных данных

In [2]:
DOWNLOAD_ROOT = "https://raw.githubusercontent.com/ageron/handson-ml/master/"
HOUSING_PATH = os.path.join("datasets", "housing")
HOUSING_URL = DOWNLOAD_ROOT + "datasets/housing/housing.tgz"

def fetch_housing_data(housing_url=HOUSING_URL, housing_path=HOUSING_PATH):
    if not os.path.isdir(housing_path):
        os.makedirs(housing_path)
    tgz_path = os.path.join(housing_path, "housing.tgz")
    urllib.request.urlretrieve(housing_url, tgz_path)
    housing_tgz = tarfile.open(tgz_path)
    housing_tgz.extractall(path=housing_path)
    housing_tgz.close()

def load_housing_data(housing_path=HOUSING_PATH):
    csv_path = os.path.join(housing_path, "housing.csv")
    return pd.read_csv(csv_path)

fetch_housing_data()
housing = load_housing_data()

/tmp/ipykernel_9425/3985674179.py:11: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  housing_tgz.extractall(path=housing_path)


# Стратифицированное разделение

In [3]:
# Добавляем категорию дохода для стратификации
housing["income_cat"] = np.ceil(housing["median_income"] / 1.5)
housing["income_cat"] = housing["income_cat"].where(housing["income_cat"] < 5, 5.0)

# Стратифицированное разбиение
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(housing, housing["income_cat"]):
    strat_train_set = housing.loc[train_index]
    strat_test_set = housing.loc[test_index]

# Отделяем метки
housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

# Заполняем пропуски медианой
median = housing["total_bedrooms"].median()
housing_num = housing.drop("ocean_proximity", axis=1)
housing_num["total_bedrooms"] = housing_num["total_bedrooms"].fillna(median)

# Готовим тестовую выборку
X_test = strat_test_set.drop("median_house_value", axis=1)
y_test = strat_test_set["median_house_value"].copy()
X_test_num = X_test.drop("ocean_proximity", axis=1)
X_test_num["total_bedrooms"] = X_test_num["total_bedrooms"].fillna(median)

# Масштабирование

In [4]:
# Масштабируем признаки
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(housing_num)
X_test_scaled = scaler.transform(X_test_num)

# Масштабируем метки
scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(housing_labels.values.reshape(-1, 1)).ravel()
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1)).ravel()

# GridSearchCV

In [5]:
param_grid = [
    {'kernel': ['linear'], 'C': [10., 30., 100., 300., 1000., 3000., 10000., 30000.]},
    {'kernel': ['rbf'], 'C': [1.0, 3.0, 10., 30., 100., 300., 1000.],
     'gamma': [0.01, 0.03, 0.1, 0.3, 1.0, 3.0]}
]

svr = SVR(max_iter=50000)
grid_search = GridSearchCV(
    svr,
    param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=3
)
grid_search.fit(X_train_scaled, y_train_scaled)

print(f"Лучшие параметры: {grid_search.best_params_}")
print(f"Лучший RMSE (CV): {np.sqrt(-grid_search.best_score_):.4f} (в масштабированных единицах)")

# Проверяем на тестовой выборке
best_grid = grid_search.best_estimator_
y_pred_scaled = best_grid.predict(X_test_scaled)
y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()

grid_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Тестовый RMSE: {grid_rmse:.2f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=50000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


Лучшие параметры: {'C': 3.0, 'gamma': 0.3, 'kernel': 'rbf'}
Лучший RMSE (CV): 0.4792 (в масштабированных единицах)
Тестовый RMSE: 53788.56


# RandomizedSearchCV

In [6]:
param_distribs = {
    'kernel': ['linear', 'rbf'],
    'C': reciprocal(20, 200000),
    'gamma': expon(scale=1.0)
}

svr = SVR(max_iter=50000)
rnd_search = RandomizedSearchCV(
    svr,
    param_distributions=param_distribs,
    n_iter=50,
    cv=5,
    scoring='neg_mean_squared_error',
    random_state=42,
    n_jobs=-1,
    verbose=2
)
rnd_search.fit(X_train_scaled, y_train_scaled)

print(f"Лучшие параметры: {rnd_search.best_params_}")
print(f"Лучший RMSE (CV): {np.sqrt(-rnd_search.best_score_):.4f} (в масштабированных единицах)")

# Проверяем на тестовой выборке
best_rnd = rnd_search.best_estimator_
y_pred_scaled = best_rnd.predict(X_test_scaled)
y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()

rnd_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Тестовый RMSE: {rnd_rmse:.2f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=50000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


Лучшие параметры: {'C': np.float64(22.76927941060928), 'gamma': np.float64(0.22169760231351215), 'kernel': 'rbf'}
Лучший RMSE (CV): 0.4809 (в масштабированных единицах)
Тестовый RMSE: 53942.77
